In [116]:
USE_LOCAL = True

In [118]:
import pandas as pd
import os
from datetime import datetime

In [122]:
file_path = "../data/health_logs.csv"

if not os.path.exists(file_path):
    df = pd.DataFrame(columns=["Date", "Weight", "Protein", "Sleep_hours", "Workout"])
    df.to_csv(file_path, index=False)
    print("File Created")
else:
    print("File Exists")


File Exists


In [4]:
date = datetime.today().strftime('%y-%m-%d')

weight = float(input("Enter weigth(kgs): "))
protein = float (input("Protein Intake(g): "))
sleep = float(input("Sleep Hours: "))
workout = int(input("Workout? (1= yes, 0= no): "))

        

Enter weigth(kgs):  56.8
Protein Intake(g):  20
Sleep Hours:  7
Workout? (1= yes, 0= no):  1


In [5]:
new_entry = pd.DataFrame([{
    "Date": date,
    "Weight": weight,
    "Protein": protein,
    "Sleep_hours": sleep,
    "Workout": workout
}])

new_entry.to_csv(file_path, mode='a', header=False, index=False)
print("Entry saved!")

Entry saved!


In [6]:
data = pd.read_csv(file_path)
data.reset_index(drop=True)
data

,Date,Weight,Protein,Sleep_hours,Workout
0,26-04-08,57.2,20.0,7.0,1
1,26-04-22,56.0,20.0,7.0,1
2,26-04-22,56.8,20.0,7.0,1


In [7]:
data = pd.read_csv("../data/health_logs.csv")
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values(by='Date')

C:\Users\rajde\AppData\Local\Temp\ipykernel_22408\1623079151.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Date'] = pd.to_datetime(data['Date'])


In [8]:
data ['Protein_required'] = data['Weight'] * 1.6
data ['Protein_score'] = data['Protein']/ data['Protein_required']
data ['sleep_score'] = data['Sleep_hours']/7
data ['workout_score'] = data['Workout']

In [9]:
data['Protein_score'] = data['Protein_score'].clip(0,1)
data['sleep_score'] = data['sleep_score'].clip(0,1)

In [10]:
data['health_score'] =(
    0.4 *data['Protein_score'] +
    0.3 * data['sleep_score'] +
    0.3 * data['workout_score']
)

In [11]:
data['health_score_10'] = (data['health_score'] *9) +1

In [12]:
recent = data.tail(7)

avg_health = recent['health_score'].mean()
avg_protein = recent['Protein'].mean()
avg_sleep = recent['Sleep_hours'].mean()
workout_days = recent['Workout'].sum()

In [13]:
def detect_patterns(data):
    recent = data.tail(7)
    issues = []

    if recent['Protein'].mean() < recent['Protein_required'].mean():
        issues.append("Consistently low protein intake")

    if recent['Sleep_hours'].mean() < 6:
        issues.append("Poor sleep pattern")

    if recent['Workout'].sum() < 3:
        issues.append("Low workout consistency")

    if recent['health_score'].mean() < 0.6:
        issues.append("Overall lifestyle needs improvement")

    return issues

In [14]:
def generate_insights(data):
    recent = data.tail(7)
    issues = detect_patterns(data)

    insight = f"""
    Last 7 Days Summary:
    Avg Protein: {recent['Protein'].mean():.1f}g
    Avg sleep: {recent['Sleep_hours'].mean():.1f}hrs
    Workout Days: {recent['Workout'].sum()}/7
    Health score: {recent['health_score_10'].mean():.1f} / 10

    Key Issues:
    {", ".join(issues) if issues else "No Major Issues"}
    """
    return insight

In [15]:
print(generate_insights(data))


    Last 7 Days Summary:
    Avg Protein: 20.0g
    Avg sleep: 7.0hrs
    Workout Days: 3/7
    Health score: 7.2 / 10

    Key Issues:
    Consistently low protein intake
    


In [16]:
import ollama

In [17]:
response = ollama.chat(
    model='phi3:mini',
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response['message']['content'])

Hello! How can I help you today?


In [18]:
knowledge = [
    "Protein intake for muscle growth should be 1.6 to 2.2 grams per kg body weight, and Optimal per meal: 0.3 – 0.5 g/kg",
    "Sleeping less than 6 hours reduces recovery and muscle growth.",
    "Consistent workouts are required to build muscle.",
    "Low protein intake can prevent muscle gain.",
    "High-quality sources: Eggs, chicken, fish Dairy (curd, paneer, milk) Plant: lentils, chickpeas, tofu",
    "Ideal sleep: 7–9 hours/night",
    "IF protein < 1.6 g/kg → suggest increasing intake",
    "IF sleep < 6 hrs → warn about recovery drop"
    "Recovery is as important as training for muscle growth."
]

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
vectorize = TfidfVectorizer()
X = vectorize.fit_transform(knowledge)

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
def retrieve_knowledge(query):
    query_vec = vectorize.transform([query])
    similarity = cosine_similarity(query_vec, X)

    index = similarity.argmax()
    return knowledge[index]

In [23]:
print(retrieve_knowledge("muscle growth"))

Sleeping less than 6 hours reduces recovery and muscle growth.


In [24]:
def get_data_insight():
    return generate_insights(data)


def get_knowledge(query):
    return retrieve_knowledge(query)

In [25]:
import chromadb

In [26]:
client = chromadb.Client()
collection = client.get_or_create_collection(name= "NutriRecall_Memory")

In [27]:
def store_memory(insight, id):
    collection.add(
        documents=[insight],
        ids=[str(id)]
    )

In [28]:
def retrieve_memory(query):
    results = collection.query(
        query_texts=[query],
        n_results=2
    )

    docs = results['documents'][0]
    

    return " \n ".join(docs)

In [30]:
insight_text = generate_insights(data)
store_memory(insight_text, id=2)

In [31]:
print(retrieve_memory("muscle growth"))


    Last 7 Days Summary:
    Avg Protein: 20.0g
    Avg sleep: 7.0hrs
    Workout Days: 3/7
    Health score: 7.2 / 10

    Key Issues:
    Consistently low protein intake
    


In [32]:
!pip install openai

In [33]:
from openai import OpenAI

In [34]:
import os
client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
    )

In [102]:
def generate_response(prompt):
    if USE_LOCAL:
        response = ollama.chat(
            model = 'phi3:mini',
            messages = [{"role":"user", "content":prompt}]
        )
        return response['message']['content']
    else:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

In [108]:
def nutrirecall_ai(query, data):

    insight = generate_insights(data)
    knowledge_piece = retrieve_knowledge(query)
    memory = retrieve_memory(query)

    prompt = f"""
You are a smart fitness and nutrition assistant, works on straight facts and nurtirional balance.

use the following tools:
- User data insights
- Nutritional Knowledge

user data:
{insight}

Relevant Knowledge:
{knowledge_piece}

Past Memory:
{memory}

User Question:
{query}

Analyze the situation step by step and give a clear, personalized, and practical answer, keep the answer short, in points minimal and direct. 
"""
    return generate_response(prompt)
    

In [110]:
USE_LOCAL = True
print(nutrirecall_ai("Why am I not gaining muscle?", data))

- Adequate protein intake is crucial for muscle repair and growth; your average of only about 20g may be insufficient, especially post-workout when recovery demand increases (reference: Nutritional Knowledge).

- Your workouts occur three times a week which might not be enough frequency or intensity to stimulate significant hypertrophy if consistently low in protein and rest days are mismanaged. Consider increasing either the volume of your sessions, their nutrient timing around intake (Nutritional Knowledge).

- Despite adequate sleep on average per day for recovery purposes at 7 hours, optimizing post-workout meal composition with higher leucine content can further enhance muscle synthesis and repair. Include foods like chicken breast or a whey protein shake within the anabolic window after exercising (User Data Insights).

- Health score of 7.2 indicates good overall health but targeted intervention in nutrition, specifically increasing daily protein intake to align with muscle reco

In [111]:
USE_LOCAL = False
print(nutrirecall_ai("Why am I not gaining muscle?", data))

Based on the provided user data and nutritional knowledge, here's an analysis of the situation:

**Reason for not gaining muscle:**

1. **Inadequate protein intake**: Averaging 20g of protein per day is below the recommended daily intake for muscle growth, which is typically 1.2-1.6g of protein per kilogram of body weight.
2. **Insufficient workout frequency**: Working out only 3 times a week may not be sufficient to stimulate muscle growth and recovery.
3. **Limited sleep**: While average sleep is 7 hours, research suggests that most adults need 7-9 hours of sleep per night for optimal muscle recovery and growth.

**Action plan:**

1. **Increase protein intake**: Aim for 1.2-1.6g of protein per kilogram of body weight per day, spread across 3-5 main meals and 2-3 snacks.
2. **Increase workout frequency**: Gradually increase workout days to 4-5 times a week, allowing for adequate rest and recovery time.
3. **Improve sleep quality**: Aim for 7-9 hours of sleep per night and establish a 